In [1]:
"""
╔══════════════════════════════════════════════════════════════╗
║  ReviewGuard — Phase 5: Trust Score Integration             ║
║  Combines Phase 2 (baseline) + Phase 3 (ML) + Phase 4       ║
║  (network) into unified product trust scores                ║
╚══════════════════════════════════════════════════════════════╝
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import pickle

warnings.filterwarnings("ignore")

# Plot settings
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# Setup directories
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

os.makedirs("outputs/dashboard", exist_ok=True)
os.makedirs("outputs/final_reports", exist_ok=True)

print("✅ Phase 5 Setup Complete!")
print(f"   Working dir: {os.getcwd()}")

✅ Phase 5 Setup Complete!
   Working dir: C:\Users\diyas\OneDrive\Desktop\reviewguard


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 1: LOAD ALL PREVIOUS PHASE OUTPUTS
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("  STEP 1: LOADING ALL PHASE OUTPUTS")
print("=" * 60)

# ── Load the master review dataset ────────────────────────────────────────────
df = pd.read_csv("data/processed/reviews_cleaned.csv", low_memory=False)
HIGH_QUALITY = ["amazon2", "amazon4", "amazon5"]
df = df[df["source_dataset"].isin(HIGH_QUALITY)].copy()
df = df.reset_index(drop=True)
df["review_id"] = range(len(df))
df["is_recommended"] = df["is_recommended"].astype(bool)

print(f"\n📊 Master dataset: {len(df):,} reviews, {df['product_id'].nunique()} products")

# ── Load Phase 2: EDA Baseline Scores ─────────────────────────────────────────
try:
    phase2_scores = pd.read_csv("data/processed/product_eda_scores.csv")
    print(f"✅ Phase 2 EDA scores loaded: {len(phase2_scores)} products")
except FileNotFoundError:
    print("⚠️ Phase 2 scores not found, creating empty")
    phase2_scores = pd.DataFrame(columns=["product_id", "eda_suspicion_score", "severity"])

# ── Load Phase 4: Network Scores ─────────────────────────────────────────────
try:
    phase4_scores = pd.read_csv("data/graphs/product_network_scores.csv")
    print(f"✅ Phase 4 network scores loaded: {len(phase4_scores)} products")
except FileNotFoundError:
    print("⚠️ Phase 4 scores not found, creating empty")
    phase4_scores = pd.DataFrame(columns=["product_id", "network_risk_score"])

# ── Load Phase 4: Suspicious Reviewers ───────────────────────────────────────
try:
    reviewer_scores = pd.read_csv("data/graphs/reviewer_network_scores.csv")
    suspicious_reviewer_ids = set(reviewer_scores["reviewer_id"].values)
    print(f"✅ Phase 4 suspicious reviewers loaded: {len(suspicious_reviewer_ids)}")
except FileNotFoundError:
    suspicious_reviewer_ids = set()
    print("⚠️ Phase 4 reviewer scores not found")

# ── Load Phase 3: ML Model & Predict on ALL Reviews ──────────────────────────
try:
    with open("models/phase3_xgboost.pkl", "rb") as f:
        xgb_model = pickle.load(f)
    with open("models/tfidf_vectorizer.pkl", "rb") as f:
        tfidf_vec = pickle.load(f)
    with open("models/scaler.pkl", "rb") as f:
        scaler = pickle.load(f)
    with open("models/feature_names.pkl", "rb") as f:
        feature_names = pickle.load(f)
    print(f"✅ Phase 3 ML model loaded (XGBoost)")
    ml_available = True
except FileNotFoundError:
    print("⚠️ Phase 3 model not found")
    ml_available = False

print(f"\n📊 All phase outputs loaded successfully!")

  STEP 1: LOADING ALL PHASE OUTPUTS

📊 Master dataset: 59,604 reviews, 89 products
✅ Phase 2 EDA scores loaded: 89 products
✅ Phase 4 network scores loaded: 51 products
✅ Phase 4 suspicious reviewers loaded: 82
✅ Phase 3 ML model loaded (XGBoost)

📊 All phase outputs loaded successfully!


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 2: APPLY PHASE 3 ML MODEL TO ALL REVIEWS
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("  STEP 2: RUNNING ML MODEL ON ALL 59K REVIEWS")
print("=" * 60)

if ml_available:
    print("\n⚙️  Extracting features for all reviews...")
    
    # ── Text features function ────────────────────────────────────────────────
    def extract_text_features(text):
        if pd.isna(text) or text == "":
            return {k: 0 for k in ["char_count", "word_count", "avg_word_length",
                    "uppercase_ratio", "exclamation_count", "question_count",
                    "punctuation_ratio", "digit_ratio", "capital_words", "unique_word_ratio"]}
        text = str(text)
        words = text.split()
        char_count = len(text)
        word_count = len(words)
        return {
            "char_count": char_count,
            "word_count": word_count,
            "avg_word_length": np.mean([len(w) for w in words]) if words else 0,
            "uppercase_ratio": sum(1 for c in text if c.isupper()) / max(char_count, 1),
            "exclamation_count": text.count("!"),
            "question_count": text.count("?"),
            "punctuation_ratio": sum(1 for c in text if c in ".,;:!?-") / max(char_count, 1),
            "digit_ratio": sum(1 for c in text if c.isdigit()) / max(char_count, 1),
            "capital_words": sum(1 for w in words if w.isupper() and len(w) > 1),
            "unique_word_ratio": len(set(words)) / max(word_count, 1)
        }
    
    # Extract text features
    text_features = df["review_text"].apply(extract_text_features)
    text_features_df = pd.DataFrame(text_features.tolist())
    df = pd.concat([df.reset_index(drop=True), text_features_df.reset_index(drop=True)], axis=1)
    
    # Product-level features
    product_stats = df.groupby("product_id").agg(
        product_total_reviews=("review_id", "count"),
        product_avg_rating=("rating", "mean"),
        product_five_star_rate=("rating", lambda x: (x == 5).sum() / len(x))
    ).reset_index()
    df = df.merge(product_stats, on="product_id", how="left")
    
    # Reviewer-level features
    reviewer_stats = df.groupby("reviewer_id").agg(
        total_reviews=("review_id", "count")
    ).reset_index()
    df = df.merge(reviewer_stats, on="reviewer_id", how="left")
    
    # Rating-based features
    df["is_five_star"] = (df["rating"] == 5).astype(int)
    df["is_one_star"] = (df["rating"] == 1).astype(int)
    df["is_extreme_rating"] = (df["rating"].isin([1, 5])).astype(int)
    df["rating_vs_product_avg"] = df["rating"] - df["product_avg_rating"]
    df["is_recommended_flag"] = df["is_recommended"].astype(int)
    df["five_star_not_recommended"] = ((df["is_five_star"] == 1) & (df["is_recommended_flag"] == 0)).astype(int)
    df["short_five_star"] = ((df["is_five_star"] == 1) & (df["word_count"] < 20)).astype(int)
    
    if "review_length" not in df.columns:
        df["review_length"] = df["review_text"].astype(str).str.len()
    if "helpful_votes" not in df.columns:
        df["helpful_votes"] = 0
    
    print(f"✅ Feature extraction complete")
    
    # ── Build feature matrix in same order as training ────────────────────────
    print(f"\n⚙️  Building feature matrix for prediction...")
    
    # Get the numeric features that XGBoost expects
    numeric_features_needed = [f for f in feature_names if not f.startswith("tfidf_")]
    
    # Fill missing columns with 0
    for feat in numeric_features_needed:
        if feat not in df.columns:
            df[feat] = 0
    
    X_numeric = df[numeric_features_needed].fillna(0)
    
    # TF-IDF features
    review_texts = df["review_text"].fillna("").astype(str)
    tfidf_matrix = tfidf_vec.transform(review_texts)
    tfidf_df = pd.DataFrame(
        tfidf_matrix.toarray(),
        columns=[f"tfidf_{name}" for name in tfidf_vec.get_feature_names_out()],
        index=df.index
    )
    
    # Combine
    X = pd.concat([X_numeric.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)
    
    # Make sure column order matches training
    X = X[feature_names]
    
    # Scale numeric features
    X_scaled = X.copy()
    X_scaled[numeric_features_needed] = scaler.transform(X[numeric_features_needed])
    
    print(f"✅ Feature matrix ready: {X_scaled.shape}")
    
    # ── Predict fake probability for every review ────────────────────────────
    print(f"\n⚙️  Running XGBoost predictions on all 59K reviews...")
    fake_probabilities = xgb_model.predict_proba(X_scaled)[:, 1]
    df["ml_fake_probability"] = fake_probabilities
    df["ml_predicted_fake"] = (fake_probabilities >= 0.5).astype(int)
    
    n_flagged = df["ml_predicted_fake"].sum()
    print(f"\n📊 ML Prediction Results:")
    print(f"   Reviews flagged as FAKE: {n_flagged:,} ({n_flagged/len(df)*100:.1f}%)")
    print(f"   Average fake probability : {fake_probabilities.mean():.3f}")
    print(f"   High confidence FAKE (>0.7): {(fake_probabilities > 0.7).sum():,}")
    print(f"   Very high confidence (>0.9): {(fake_probabilities > 0.9).sum():,}")
else:
    df["ml_fake_probability"] = 0
    df["ml_predicted_fake"] = 0
    print("⚠️ ML model not available, using 0 for all reviews")

print(f"\n✅ Phase 3 ML scores computed for all reviews")

  STEP 2: RUNNING ML MODEL ON ALL 59K REVIEWS

⚙️  Extracting features for all reviews...
✅ Feature extraction complete

⚙️  Building feature matrix for prediction...
✅ Feature matrix ready: (59604, 126)

⚙️  Running XGBoost predictions on all 59K reviews...

📊 ML Prediction Results:
   Reviews flagged as FAKE: 19,840 (33.3%)
   Average fake probability : 0.360
   High confidence FAKE (>0.7): 12,792
   Very high confidence (>0.9): 4,588

✅ Phase 3 ML scores computed for all reviews


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 3: COMBINE ALL SIGNALS INTO UNIFIED TRUST SCORE
#  Formula: Weight each phase's signal by reliability
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("  STEP 3: COMPUTING UNIFIED TRUST SCORE PER PRODUCT")
print("=" * 60)

print(f"\n⚙️  Aggregating signals for all 89 products...")

# ── Compute per-product metrics ──────────────────────────────────────────────
product_analysis = df.groupby("product_id").agg(
    product_name=("product_name", "first"),
    total_reviews=("review_id", "count"),
    avg_rating=("rating", "mean"),
    five_star_count=("rating", lambda x: (x == 5).sum()),
    recommended_count=("is_recommended", "sum"),
    ml_fake_count=("ml_predicted_fake", "sum"),
    ml_avg_fake_prob=("ml_fake_probability", "mean"),
    ml_high_confidence_fake=("ml_fake_probability", lambda x: (x > 0.7).sum())
).reset_index()

product_analysis["five_star_rate"] = product_analysis["five_star_count"] / product_analysis["total_reviews"]
product_analysis["recommendation_rate"] = product_analysis["recommended_count"] / product_analysis["total_reviews"]
product_analysis["ml_fake_rate"] = product_analysis["ml_fake_count"] / product_analysis["total_reviews"]

# ── Merge Phase 2 EDA scores ─────────────────────────────────────────────────
product_analysis = product_analysis.merge(
    phase2_scores[["product_id", "eda_suspicion_score"]].rename(
        columns={"eda_suspicion_score": "phase2_score"}
    ),
    on="product_id", how="left"
)
product_analysis["phase2_score"] = product_analysis["phase2_score"].fillna(0)

# ── Merge Phase 4 network scores ─────────────────────────────────────────────
product_analysis = product_analysis.merge(
    phase4_scores[["product_id", "network_risk_score"]].rename(
        columns={"network_risk_score": "phase4_score"}
    ),
    on="product_id", how="left"
)
product_analysis["phase4_score"] = product_analysis["phase4_score"].fillna(0)

# ── Phase 3 ML score per product (0-100 scale) ───────────────────────────────
# Convert average fake probability to 0-100 scale
product_analysis["phase3_score"] = (product_analysis["ml_avg_fake_prob"] * 100).clip(upper=100)

# ── UNIFIED TRUST SCORE FORMULA ──────────────────────────────────────────────
# Weighted combination of all 3 signals
# Weights based on signal reliability:
#   Phase 2 (heuristic): 30% - Well-established statistical baseline
#   Phase 3 (ML):        30% - Learned patterns from ground truth
#   Phase 4 (network):   40% - Strongest coordination detection

W_PHASE2 = 0.30
W_PHASE3 = 0.30
W_PHASE4 = 0.40

product_analysis["suspicion_score"] = (
    product_analysis["phase2_score"] * W_PHASE2 +
    product_analysis["phase3_score"] * W_PHASE3 +
    product_analysis["phase4_score"] * W_PHASE4
).clip(0, 100)

# ── TRUST SCORE = INVERSE OF SUSPICION ───────────────────────────────────────
product_analysis["trust_score"] = 100 - product_analysis["suspicion_score"]

# ── Classify trust level ─────────────────────────────────────────────────────
def classify_trust(score):
    if score >= 80:
        return "HIGH TRUST"
    elif score >= 60:
        return "MEDIUM TRUST"
    elif score >= 40:
        return "LOW TRUST"
    else:
        return "CRITICAL RISK"

product_analysis["trust_level"] = product_analysis["trust_score"].apply(classify_trust)

# ── Sort by trust score (lowest first = most suspicious) ────────────────────
product_analysis = product_analysis.sort_values("trust_score")

# ── Print summary ────────────────────────────────────────────────────────────
print(f"\n📊 Trust Score Distribution:")
trust_dist = product_analysis["trust_level"].value_counts()
for level in ["HIGH TRUST", "MEDIUM TRUST", "LOW TRUST", "CRITICAL RISK"]:
    count = trust_dist.get(level, 0)
    pct = count / len(product_analysis) * 100
    emoji = {"HIGH TRUST": "🟢", "MEDIUM TRUST": "🟡", 
             "LOW TRUST": "🟠", "CRITICAL RISK": "🔴"}[level]
    print(f"   {emoji} {level:<15} {count:2d} products ({pct:.1f}%)")

print(f"\n📊 Trust Score Statistics:")
print(f"   Mean trust score : {product_analysis['trust_score'].mean():.1f}")
print(f"   Median           : {product_analysis['trust_score'].median():.1f}")
print(f"   Highest trust    : {product_analysis['trust_score'].max():.1f}")
print(f"   Lowest trust     : {product_analysis['trust_score'].min():.1f}")

# ── Show top 10 CRITICAL products ────────────────────────────────────────────
print(f"\n🚨 TOP 10 MOST SUSPICIOUS PRODUCTS:")
for idx, row in product_analysis.head(10).iterrows():
    name = str(row["product_name"])[:45]
    print(f"\n   [{row['trust_level']:>13}] {name}")
    print(f"      Trust: {row['trust_score']:5.1f}/100 | Reviews: {row['total_reviews']} | Rating: {row['avg_rating']:.2f}⭐")
    print(f"      Phase 2: {row['phase2_score']:5.1f} | Phase 3: {row['phase3_score']:5.1f} | Phase 4: {row['phase4_score']:5.1f}")

  STEP 3: COMPUTING UNIFIED TRUST SCORE PER PRODUCT

⚙️  Aggregating signals for all 89 products...

📊 Trust Score Distribution:
   🟢 HIGH TRUST      21 products (23.6%)
   🟡 MEDIUM TRUST    51 products (57.3%)
   🟠 LOW TRUST       12 products (13.5%)
   🔴 CRITICAL RISK    5 products (5.6%)

📊 Trust Score Statistics:
   Mean trust score : 69.9
   Median           : 76.1
   Highest trust    : 96.6
   Lowest trust     : 32.9

🚨 TOP 10 MOST SUSPICIOUS PRODUCTS:

   [CRITICAL RISK] All-New Kindle Oasis E-reader - 7 High-Resolu
      Trust:  32.9/100 | Reviews: 22 | Rating: 4.59⭐
      Phase 2:  35.0 | Phase 3:  55.2 | Phase 4: 100.0

   [CRITICAL RISK] Fire TV Stick Streaming Media Player Pair Kit
      Trust:  34.9/100 | Reviews: 6 | Rating: 5.00⭐
      Phase 2:  60.0 | Phase 3:  23.8 | Phase 4: 100.0

   [CRITICAL RISK] All-New Kindle Oasis E-reader - 7 High-Resolu
      Trust:  35.0/100 | Reviews: 7 | Rating: 4.43⭐
      Phase 2:  29.5 | Phase 3:  53.8 | Phase 4: 100.0

   [CRITICAL RIS

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 4: CALCULATE ADJUSTED RATINGS
#  Remove suspicious reviews and recalculate "true" rating
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("  STEP 4: COMPUTING ADJUSTED RATINGS")
print("=" * 60)

# For each review, determine if it's "suspicious"
# A review is suspicious if:
#   1. ML flagged it as fake (ml_predicted_fake == 1), OR
#   2. Reviewer is in suspicious network (Phase 4)
df["is_suspicious_review"] = (
    (df["ml_predicted_fake"] == 1) |
    (df["reviewer_id"].isin(suspicious_reviewer_ids))
).astype(int)

# Count suspicious reviews per product
suspicious_by_product = df.groupby("product_id").agg(
    suspicious_count=("is_suspicious_review", "sum")
).reset_index()

product_analysis = product_analysis.merge(suspicious_by_product, on="product_id", how="left")
product_analysis["suspicious_count"] = product_analysis["suspicious_count"].fillna(0).astype(int)
product_analysis["clean_review_count"] = product_analysis["total_reviews"] - product_analysis["suspicious_count"]
product_analysis["pct_suspicious"] = (product_analysis["suspicious_count"] / product_analysis["total_reviews"] * 100).round(1)

# Calculate adjusted rating (average of only non-suspicious reviews)
adjusted_ratings = []
for pid in product_analysis["product_id"]:
    clean_reviews = df[(df["product_id"] == pid) & (df["is_suspicious_review"] == 0)]
    if len(clean_reviews) > 0:
        adjusted_ratings.append(clean_reviews["rating"].mean())
    else:
        # If ALL reviews are suspicious, keep original
        adjusted_ratings.append(product_analysis[product_analysis["product_id"] == pid]["avg_rating"].iloc[0])

product_analysis["adjusted_rating"] = adjusted_ratings
product_analysis["rating_inflation"] = product_analysis["avg_rating"] - product_analysis["adjusted_rating"]
product_analysis["rating_inflation"] = product_analysis["rating_inflation"].round(2)
product_analysis["adjusted_rating"] = product_analysis["adjusted_rating"].round(2)

# ── Summary statistics ──────────────────────────────────────────────────────
print(f"\n📊 Rating Adjustment Analysis:")
print(f"   Products where rating dropped (rating_inflation > 0):")
n_dropped = (product_analysis["rating_inflation"] > 0).sum()
print(f"      {n_dropped} products ({n_dropped/len(product_analysis)*100:.1f}%)")

print(f"\n   Average rating inflation: {product_analysis['rating_inflation'].mean():.2f}⭐")
print(f"   Maximum rating inflation: {product_analysis['rating_inflation'].max():.2f}⭐")

print(f"\n📊 Total Review Statistics:")
print(f"   Total reviews analyzed  : {product_analysis['total_reviews'].sum():,}")
print(f"   Total suspicious reviews: {product_analysis['suspicious_count'].sum():,}")
print(f"   Suspicious rate         : {product_analysis['suspicious_count'].sum()/product_analysis['total_reviews'].sum()*100:.1f}%")

# ── Show top 10 rating adjustments ─────────────────────────────────────────
print(f"\n🚨 TOP 10 PRODUCTS WITH BIGGEST RATING INFLATION:")
top_inflation = product_analysis.nlargest(10, "rating_inflation")
for idx, row in top_inflation.iterrows():
    name = str(row["product_name"])[:40]
    print(f"\n   {name}")
    print(f"      Displayed: {row['avg_rating']:.2f}⭐ → Adjusted: {row['adjusted_rating']:.2f}⭐ (drop {row['rating_inflation']:.2f}⭐)")
    print(f"      Total: {row['total_reviews']} | Suspicious: {row['suspicious_count']} ({row['pct_suspicious']:.1f}%)")
    print(f"      Trust Score: {row['trust_score']:.1f}/100 | {row['trust_level']}")

  STEP 4: COMPUTING ADJUSTED RATINGS

📊 Rating Adjustment Analysis:
   Products where rating dropped (rating_inflation > 0):
      31 products (34.8%)

   Average rating inflation: -0.02⭐
   Maximum rating inflation: 1.12⭐

📊 Total Review Statistics:
   Total reviews analyzed  : 59,604
   Total suspicious reviews: 20,556
   Suspicious rate         : 34.5%

🚨 TOP 10 PRODUCTS WITH BIGGEST RATING INFLATION:

   Echo (White),,,
Echo (White),,,
      Displayed: 3.12⭐ → Adjusted: 2.00⭐ (drop 1.12⭐)
      Total: 8 | Suspicious: 4 (50.0%)
      Trust Score: 83.0/100 | HIGH TRUST

   Unknown Product
      Displayed: 3.92⭐ → Adjusted: 3.00⭐ (drop 0.92⭐)
      Total: 13 | Suspicious: 10 (76.9%)
      Trust Score: 72.9/100 | MEDIUM TRUST

   Unknown Product
      Displayed: 3.07⭐ → Adjusted: 2.38⭐ (drop 0.69⭐)
      Total: 15 | Suspicious: 7 (46.7%)
      Trust Score: 83.2/100 | HIGH TRUST

   Amazon Kindle Replacement Power Adapter 
      Displayed: 2.80⭐ → Adjusted: 2.33⭐ (drop 0.47⭐)
      Tota

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 5: SAVE ALL DATA FOR STREAMLIT DASHBOARD
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("  STEP 5: SAVING DASHBOARD DATA")
print("=" * 60)

# ── Final product report ────────────────────────────────────────────────────
final_columns = [
    "product_id", "product_name", 
    "trust_score", "trust_level",
    "avg_rating", "adjusted_rating", "rating_inflation",
    "total_reviews", "suspicious_count", "clean_review_count", "pct_suspicious",
    "five_star_rate", "recommendation_rate",
    "phase2_score", "phase3_score", "phase4_score",
    "ml_fake_count", "ml_high_confidence_fake"
]

final_report = product_analysis[final_columns].copy()

# Round for cleanliness
for col in ["trust_score", "phase2_score", "phase3_score", "phase4_score"]:
    final_report[col] = final_report[col].round(1)
for col in ["avg_rating", "adjusted_rating", "rating_inflation", 
            "five_star_rate", "recommendation_rate"]:
    final_report[col] = final_report[col].round(2)

# Save to CSV
final_report.to_csv("outputs/dashboard/product_trust_report.csv", index=False)
print(f"\n✅ Saved: outputs/dashboard/product_trust_report.csv ({len(final_report)} products)")

# ── Individual review-level data (for dashboard drill-down) ────────────────
review_report = df[[
    "review_id", "product_id", "product_name", "reviewer_id", "reviewer_name",
    "rating", "review_date", "review_text", "is_recommended",
    "ml_fake_probability", "ml_predicted_fake", "is_suspicious_review"
]].copy()

review_report["ml_fake_probability"] = review_report["ml_fake_probability"].round(3)

review_report.to_csv("outputs/dashboard/review_details.csv", index=False)
print(f"✅ Saved: outputs/dashboard/review_details.csv ({len(review_report):,} reviews)")

# ── Summary statistics for dashboard ───────────────────────────────────────
summary_stats = {
    "total_products": len(final_report),
    "total_reviews": int(final_report["total_reviews"].sum()),
    "suspicious_reviews": int(final_report["suspicious_count"].sum()),
    "suspicion_rate": float(round(final_report["suspicious_count"].sum() / final_report["total_reviews"].sum() * 100, 1)),
    "high_trust_products": int((final_report["trust_level"] == "HIGH TRUST").sum()),
    "medium_trust_products": int((final_report["trust_level"] == "MEDIUM TRUST").sum()),
    "low_trust_products": int((final_report["trust_level"] == "LOW TRUST").sum()),
    "critical_products": int((final_report["trust_level"] == "CRITICAL RISK").sum()),
    "avg_trust_score": float(round(final_report["trust_score"].mean(), 1)),
    "avg_rating_inflation": float(round(final_report["rating_inflation"].mean(), 2)),
    "max_rating_inflation": float(round(final_report["rating_inflation"].max(), 2))
}

import json
with open("outputs/dashboard/summary_stats.json", "w") as f:
    json.dump(summary_stats, f, indent=2)
print(f"✅ Saved: outputs/dashboard/summary_stats.json")

print(f"\n📊 FINAL DASHBOARD DATA READY:")
print(f"   Products: {summary_stats['total_products']}")
print(f"   Reviews : {summary_stats['total_reviews']:,}")
print(f"   Critical risk products: {summary_stats['critical_products']}")
print(f"   Average trust score  : {summary_stats['avg_trust_score']}/100")
print(f"   Avg rating inflation : {summary_stats['avg_rating_inflation']}⭐")

print(f"\n🎯 Data is now ready for Streamlit dashboard!")

  STEP 5: SAVING DASHBOARD DATA

✅ Saved: outputs/dashboard/product_trust_report.csv (89 products)
✅ Saved: outputs/dashboard/review_details.csv (59,604 reviews)
✅ Saved: outputs/dashboard/summary_stats.json

📊 FINAL DASHBOARD DATA READY:
   Products: 89
   Reviews : 59,604
   Critical risk products: 5
   Average trust score  : 69.9/100
   Avg rating inflation : -0.02⭐

🎯 Data is now ready for Streamlit dashboard!
